In [1]:
import sys
sys.path.append('..')  # Adds parent directory to sys.path
import pickle
import os
from tqdm import tqdm

# Get seq dict

## HG

In [ ]:
all_seq_dicts = {}
dist = '8A'
sample_type = 'random_mut'  # unmut, swapped, dimer, swapped_abag, capri, preppi, random_mut
pdb_file = f'../../GraPPI_data/curated_db/{sample_type}_sthg_{dist}'
save_dir = f'../../GraPPI_data/curated_db/'
pdbs = [f for f in os.listdir(pdb_file) if f.endswith('.pkl')]

for pdb in tqdm(pdbs):
    a_seq_dict = {'receptor': {'seq': None}, 'ligand': {'seq': None}}
    with open(os.path.join(pdb_file, pdb), 'rb') as f:
        one_sthg = pickle.load(f)
    for part in ['receptor', 'ligand']:
        a_seq_dict[part]['seq'] = ''.join(one_sthg.AA_df[one_sthg.AA_df['chain_type'] == part]['res_letter'].tolist())
    all_seq_dicts[pdb[:-4]] = a_seq_dict
pdb_list = list(all_seq_dicts.keys())
print(len(pdb_list))
with open(f'{save_dir}{sample_type}_seq_dicts_{dist}_hg.pkl', 'wb') as f:
    pickle.dump(all_seq_dicts, f)

100%|██████████| 8555/8555 [00:22<00:00, 381.07it/s]

8555


For test sets

In [ ]:
all_seq_dicts = {}
test_types = ['S90', 'S79', 'SKEMPI_test']
save_dir = f'../../GraPPI_data/curated_db/'
S79_lst = []
S90_lst = []
Skempi_lst = []
for test_type in test_types:
	pdb_file = f'../../GraPPI_data/curated_db/{test_type}'
	pdbs = [f for f in os.listdir(pdb_file) if f.endswith('.pkl')]
	if test_type == 'S79':
		S79_lst.extend(pdbs)
	elif test_type == 'S90':
		S90_lst.extend(pdbs)
	elif test_type == 'SKEMPI_test':
		Skempi_lst.extend(pdbs)

	for pdb in tqdm(pdbs):
		a_seq_dict = {'receptor': {'seq': None}, 'ligand': {'seq': None}}
		with open(os.path.join(pdb_file, pdb), 'rb') as f:
			one_sthg = pickle.load(f)
		for part in ['receptor', 'ligand']:
			a_seq_dict[part]['seq'] = ''.join(one_sthg.AA_df[one_sthg.AA_df['chain_type'] == part]['res_letter'].tolist())
		all_seq_dicts[pdb[:-4]] = a_seq_dict
pdb_list = list(all_seq_dicts.keys())
with open(f'{save_dir}{sample_type}_seq_dicts_{dist}_hg_test.pkl', 'wb') as f:
    pickle.dump(all_seq_dicts, f)


100%|██████████| 166/166 [00:00<00:00, 383.82it/s]


In [4]:
len(all_seq_dicts.keys())

3041

# Get seq dict for mutants

In [ ]:
all_seq_dicts = {}
dist = '8A'
pdb_file = f'../../GraPPI_data/curated_db/mutant_sthg_{dist}'
save_dir = f'../../GraPPI_data/curated_db/'
pdbs = [f for f in os.listdir(pdb_file) if f.endswith('.pkl')]

for pdb in tqdm(pdbs):
    a_seq_dict = {'receptor': {'seq': None}, 'ligand': {'seq': None}}
    with open(os.path.join(pdb_file, pdb), 'rb') as f:
        one_sthg = pickle.load(f)
    for part in ['receptor', 'ligand']:
        a_seq_dict[part]['seq'] = ''.join(one_sthg.AA_df[one_sthg.AA_df['chain_type'] == part]['res_letter'].tolist())
    all_seq_dicts[pdb.split('.')[0]] = a_seq_dict
pdb_list = list(all_seq_dicts.keys())
print(len(pdb_list))
with open(f'{save_dir}mut_seq_dicts_{dist}_hg.pkl', 'wb') as f:
    pickle.dump(all_seq_dicts, f)

100%|██████████| 6717/6717 [18:13<00:00,  6.14it/s]  

6717


# Perform ESM to get embeddings of seqs

In [ ]:
import torch
import esm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if_test = False # to get test list, need to run all the way from the beginning
try:
    if_mut
except NameError:
	if_mut = False
try:
    sample_type
except NameError:
	sample_type = 'random_mut'  # unmut, swapped, dimer, swapped_abag, preppi, random_mut
dist = '8A'
esm_dim = 1280 # 480 1280
if esm_dim == 1280:
	esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
elif esm_dim == 480:
	esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
batch_converter = alphabet.get_batch_converter()
esm_model = esm_model.to(device)
esm_model.eval()
if if_mut:
	seq_dic_path = f'../../GraPPI_data/curated_db/mut_seq_dicts_{dist}_hg.pkl'
elif not if_mut:
    seq_dic_path = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_{dist}_hg.pkl'
else:
    seq_dic_path = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_{dist}_hg_test.pkl'
try:
	all_seq_dicts
except NameError:
	with open(seq_dic_path, 'rb') as f:
		all_seq_dicts = pickle.load(f)

In [3]:
layer_id = esm_model.num_layers
print(f'Extracting ESM-2 representations from layer {layer_id} for all sequences...')
for pdb in tqdm(all_seq_dicts.keys()):
    interface_emb = []
    for part in ['receptor', 'ligand']:
        seq = all_seq_dicts[pdb][part]['seq']
        raw_input = [("p", seq)]
        _, _, batch_tokens = batch_converter(raw_input)
        batch_tokens = batch_tokens.to(device)
        with torch.no_grad():
            results = esm_model(batch_tokens, repr_layers=[layer_id], return_contacts=False)
            token_embeddings = results["representations"][layer_id]
            output = token_embeddings[0,1:-1,:].cpu()  # Move back to CPU for storage
        all_seq_dicts[pdb][part]['seq_emb'] = output

Extracting ESM-2 representations from layer 12 for all sequences...


100%|██████████| 8555/8555 [01:30<00:00, 94.12it/s]


In [4]:
print(layer_id)

12


In [5]:
len(all_seq_dicts[pdb]['receptor']['seq']), len(all_seq_dicts[pdb]['ligand']['seq']), all_seq_dicts[pdb]['receptor']['seq_emb'].shape, all_seq_dicts[pdb]['ligand']['seq_emb'].shape

(148, 148, torch.Size([148, 480]), torch.Size([148, 480]))

In [ ]:
if if_mut:
    if esm_dim == 1280:
        save_dir = f'../../GraPPI_data/curated_db/mut_seq_dicts_esm_{dist}_hg.pkl'
    else:
        save_dir = f'../../GraPPI_data/curated_db/mut_seq_dicts_esm{esm_dim}_{dist}_hg.pkl'
else:
    if esm_dim == 1280:
        if if_test:
            save_dir = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_esm_{dist}_hg_test.pkl'
        else:
            save_dir = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_esm_{dist}_hg.pkl'
    else:
        if if_test:
            save_dir = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_esm{esm_dim}_{dist}_hg_test.pkl'
        else:
            save_dir = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_esm{esm_dim}_{dist}_hg.pkl'
with open(f'{save_dir}', 'wb') as f:
    pickle.dump(all_seq_dicts, f)

# Load new seq dict with ESM emb

In [ ]:
import torch
model_select = 'esm'
try:
    sample_type
except NameError:
	sample_type = 'random_mut'  # unmut, swapped, dimer, swapped_abag, preppi, random_mut
dist = '8A'
try:
    if_mut
except NameError:
	if_mut = False
try:
    if_test
except NameError:
	if_test = False
try:
    esm_dim
except NameError:
	esm_dim = 1280 # 480 1280

In [ ]:
if if_mut:
    try:
        pdb_file
    except NameError:
        pdb_file = f'../../curated_db/mutant_sthg_{dist}'
    if model_select == 'esm':
        if esm_dim == 1280:
            dict_dir = f'../../GraPPI_data/curated_db/mut_seq_dicts_esm_{dist}_hg.pkl'
            save_dir = f'../../GraPPI_data/curated_db/mutant_sthg_esm_{dist}'
        else:
            dict_dir = f'../../GraPPI_data/curated_db/mut_seq_dicts_esm{esm_dim}_{dist}_hg.pkl'
            save_dir = f'../../GraPPI_data/curated_db/mutant_sthg_esm{esm_dim}_{dist}'
else:
    try:
        pdb_file
    except NameError:
        pdb_file = f'../../GraPPI_data/curated_db/{sample_type}_sthg_{dist}'
    if model_select == 'esm':
        if esm_dim == 1280:
            dict_dir = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_esm_{dist}_hg.pkl'
            save_dir = f'../../GraPPI_data/curated_db/{sample_type}_sthg_esm_{dist}'
        else:
            dict_dir = f'../../GraPPI_data/curated_db/{sample_type}_seq_dicts_esm{esm_dim}_{dist}_hg.pkl'
            save_dir = f'../../GraPPI_data/curated_db/{sample_type}_sthg_esm{esm_dim}_{dist}'
print(save_dir)

../../curated_db/random_mut_sthg_esm480_8A


In [9]:
try:
    all_seq_dicts
except NameError:
    with open(f'{dict_dir}', 'rb') as f:
        all_seq_dicts = pickle.load(f)

In [10]:
print(f'Choose: {model_select};\nPDB file: {pdb_file};\nDictionary file: {dict_dir};\n Save directory: {save_dir}')
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

Choose: esm;
PDB file: ../../curated_db/random_mut_sthg_8A;
Dictionary file: /mnt/boltzmann/data/zhisong/curated_db/random_mut_seq_dicts_esm480_8A_hg.pkl;
 Save directory: ../../curated_db/random_mut_sthg_esm480_8A


In [11]:
pdb_fs = [f for f in os.listdir(pdb_file) if f.endswith('.pkl')]
#for pdb in tqdm(all_seq_dicts.keys()):
for pdb_f in tqdm(pdb_fs):
    pdb = pdb_f[:-4]  # Remove .pkl extension
    if pdb == '1OAU' or pdb == '1OAU'.lower():
        continue
    with open(os.path.join(pdb_file, f'{pdb}.pkl'), 'rb') as f:
        one_sthg = pickle.load(f)
    for part in ['receptor', 'ligand']:
        one_sthg.protein_graph[part].x = torch.cat((all_seq_dicts[pdb][part]['seq_emb'],one_sthg.protein_graph[part].x[:,20:]),dim=1)
    with open(os.path.join(save_dir, f'{pdb}.pkl'), 'wb') as f:
        pickle.dump(one_sthg, f)

100%|██████████| 8555/8555 [00:40<00:00, 213.59it/s]


In [12]:
one_sthg.protein_graph

HeteroData(
  y=0.0,
  mut_info='A_Q56R,A_D79K,A_I80V,A_R88P,A_W94P,B_E59Y,B_T83P,B_G89F,B_L95G,B_K102T',
  pdb_name='2qf9',
  ligand={ x=[148, 485] },
  receptor={ x=[148, 485] },
  (receptor, receptor_receptor, receptor)={
    edge_index=[2, 3048],
    edge_attr=[3048, 8],
  },
  (ligand, ligand_ligand, ligand)={
    edge_index=[2, 3074],
    edge_attr=[3074, 8],
  },
  (receptor, receptor_ligand, ligand)={
    edge_index=[2, 202],
    edge_attr=[202, 8],
  },
  (ligand, ligand_receptor, receptor)={
    edge_index=[2, 202],
    edge_attr=[202, 8],
  }
)

# For test

In [24]:
try:
    test_types
except NameError:
    test_types = ['S90', 'S79', 'SKEMPI_test']
for test_type in test_types:
	pdb_file = f'../../curated_db/{test_type}'
	pdb_fs = [f for f in os.listdir(pdb_file) if f.endswith('.pkl')]
	save_dir = f'../../curated_db/{test_type}_esm{esm_dim if esm_dim == 480 else ""}'
	#for pdb in tqdm(all_seq_dicts.keys()):
	for pdb_f in tqdm(pdb_fs):
		pdb = pdb_f[:-4]  # Remove .pkl extension
		with open(os.path.join(pdb_file, f'{pdb}.pkl'), 'rb') as f:
			one_sthg = pickle.load(f)
		for part in ['receptor', 'ligand']:
			one_sthg.protein_graph[part].x = torch.cat((all_seq_dicts[pdb][part]['seq_emb'],one_sthg.protein_graph[part].x[:,20:]),dim=1)
		with open(os.path.join(save_dir, f'{pdb}.pkl'), 'wb') as f:
			pickle.dump(one_sthg, f)

100%|██████████| 166/166 [00:00<00:00, 250.12it/s]
